## 1. Conexão com o Drive e instalação de dependências
Monta o Drive para acessar `dados/` e salvar artefatos e `resultados/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.2 — Dependências e imports

Instala `peft`, `transformers`, `scikit-learn` e utilitários. Todos os imports são feitos uma única vez aqui para evitar dependências implícitas entre células.

In [ ]:
!pip install -q --upgrade "torchao>=0.16.0"
!pip install -q scikit-learn peft transformers accelerate openpyxl xlsxwriter ftfy
import os, gc, json, time, random, html, re, unicodedata, ftfy
from datetime import datetime
from pathlib import Path
from collections import Counter
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import StratifiedGroupKFold,StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score, accuracy_score, matthews_corrcoef,
    classification_report, confusion_matrix,
)
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType
import hashlib
import torch.nn.functional as F
from collections import Counter, defaultdict

## 1.3 — Verificação de hardware

Confirma que o runtime tem GPU disponível. Sem CUDA, o treino cai para CPU e fica ~50× mais lento.

In [ ]:
print("CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Torch:", torch.__version__)

CUDA: True
Device: NVIDIA RTX PRO 6000 Blackwell Server Edition
Torch: 2.11.0+cu130


## 2 — Configuração global

- `SEED=42`: garante reprodutibilidade entre execuções.
- `CFG_TREINO`: hiperparâmetros do BERT+LoRA (modelo, batch, LR, LoRA, label smoothing). Único ponto de mudança para experimentos de hiperparâmetros.
- `N_FOLDS=10`, `FOLD=0`: configuração da validação cruzada. Aqui rodamos só o fold 0 para triagem rápida.
- `MIN_CHARS=30`: descarta respostas muito curtas (ruído de digitação, "ok", etc.).



In [ ]:
SEED        = 42
N_FOLDS     = 10
FOLD        = 0
MIN_CHARS   = 30
CLASSES     = ["c1", "c234", "c5"]
class_to_idx = {c: i for i, c in enumerate(CLASSES)}
WARMUP_PCT  = 0.10
MAX_LENGTH=384

CFG_TREINO = dict(
    model="neuralmind/bert-base-portuguese-cased",
    batch=16, grad_accum=1, epochs=5, max_length=384,
    lora_r=32, lora_alpha=64, lora_dropout=0.1,
    target_modules=["query","key","value","output.dense","intermediate.dense"],
    lr_lora=3e-5, lr_clf=3e-4, label_smoothing=0.05,
)

DRIVE_ROOT = Path("/content/drive/MyDrive/EP_NLP")
TRAIN_PATH = DRIVE_ROOT / "dados" / "train.xlsx"
OUT_DIR    = DRIVE_ROOT / "resultados" / "v8_variantes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 3 — Utilitários

- `set_seed`: fixa todas as seeds (Python, NumPy, PyTorch, CUDA) para reprodutibilidade.
- `Logger`: escreve no stdout e em arquivo simultaneamente, para preservar histórico entre sessões do Colab.
- `normalizar_grupo`: chave de agrupamento para `StratifiedGroupKFold`. Usa apenas os primeiros 200 caracteres normalizados — captura duplicatas e near-duplicates que começam igual.
- `hash_grupo`: variante estrita — hash do texto inteiro normalizado. Só agrupa duplicatas exatas.
- `extract/load_trainable_state`: serializa apenas LoRA + classifier, deixando o backbone BERT intacto (economiza ~99% do espaço).

In [ ]:
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class Logger:
    def __init__(self, path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._f = open(self.path, "a", encoding="utf-8")
    def __call__(self, msg=""):
        print(msg)
        self._f.write(str(msg) + "\n")
        self._f.flush()
    def close(self):
        self._f.close()

def normalizar_grupo(s: str, prefix: int = 200) -> str:
    """Chave de grupo para StratifiedGroupKFold.
    Deve ser estável e capturar duplicatas exatas/quase-exatas."""
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("utf-8")
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", "", s).strip()
    return s[:prefix]

def hash_grupo(s: str) -> str:
    if not isinstance(s, str): return ""
    s = s.lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("utf-8")
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", "", s).strip()
    return hashlib.md5(s.encode()).hexdigest()

## 4 — Pré-processamento de texto

Pipeline de limpeza antes de tokenizar:
1. `corrigir_encoding`: conserta mojibake (UTF-8 mal interpretado) via `ftfy`, resolve entidades HTML (`&amp;`, `&nbsp;`), remove caracteres de controle.
2. `limpar_saudacao_rodape`: remove saudações ("Prezado Senhor", "Bom dia") e despedidas ("Atenciosamente", "Att."). Essas partes não carregam sinal de clareza e adicionam ruído.
3. `mascarar_entidades`: substitui URLs, e-mails, telefones, NUP, CPF, CNPJ, datas por placeholders. Evita que o modelo memorize valores específicos e generaliza melhor.
4. `limpar_texto`: pipeline acima + normalização de espaços.

In [ ]:
SAUDACAO_RE = re.compile(
    r'^\s*(?:'
    r'Prezad[oa](?:\s*\(a\))?'
    r'(?:\s+(?:Senhor(?:a)?|Sr\.?|Sra\.?|Srs\.?|Senhores|Cidadã[o]?|Caro\(a\)|Caro|Cara))?'
    r'|Caro(?:s)?(?:\s+(?:Senhor(?:es)?|Sr\.?|Sra\.?|Colega))?'
    r'|Senhor(?:a|es)?'
    r'|Sr\.?|Sra\.?'
    r'|Ilmo\.?\s*|Ilma\.?\s*'
    r'|Excelent[íi]ssim[oa]\s+Senhor(?:a)?'
    r'|Dign[íi]ssim[oa]'
    r'|Ol[áa]'
    r')'
    r'[^\n]{0,120}?[,\!\.:]\s*',
    re.IGNORECASE,
)
BOM_DIA_RE = re.compile(r'^\s*(?:Bom dia|Boa tarde|Boa noite)[!.,\s]*', re.IGNORECASE)
FECHO_RE = re.compile(
    r'(?i)(?:^|[\s\.\,\!\?\;\:\)\]])\s*'
    r'(?:Atenciosamente|Cordialmente|Respeitosamente|Sinceramente'
    r'|Abra[çc]os?|Att\.?|At\.te'
    r'|Sem\s+mais\s+para\s+o\s+momento)\b'
)

def corrigir_encoding(t):
    if not isinstance(t, str):
        return ""
    t = ftfy.fix_encoding(t)
    t = html.unescape(t)
    t = re.sub(r"&deg,?",   "º",  t)
    t = re.sub(r"&bull,?",  "•",  t)
    t = re.sub(r"&hellip,?", "...", t)
    t = re.sub(r"&[a-zA-Z]+;", " ", t)
    t = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", t)
    t = re.sub(r"[\u0091-\u0094\u0096\u0097]", "", t)
    t = t.replace("\u00a0", " ")
    t = re.sub(r"[\u200b-\u200d\ufeff]", "", t)
    return t

def mascarar_entidades(t):
    t = re.sub(r"https?://\S+|www\.\S+", " [URL] ", t)
    t = re.sub(r"\S+@\S+", " [EMAIL] ", t)
    t = re.sub(r"\(?\d{2}\)?\s?\d{4,5}[-\s]?\d{4}", " [TELEFONE] ", t)
    t = re.sub(r"\d{5}\.\d{6}/\d{4}-\d{2}", " [NUP] ", t)
    t = re.sub(r"\d{2}/\d{2}/\d{4}", " [DATA] ", t)
    t = re.sub(r"\d{3}\.\d{3}\.\d{3}-\d{2}", " [CPF] ", t)
    t = re.sub(r"\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}", " [CNPJ] ", t)
    t = re.sub(r"\b\d{6,}\b", " [NUM] ", t)
    return t

def limpar_saudacao_rodape(t):
    if not isinstance(t, str) or not t:
        return t
    t = SAUDACAO_RE.sub('', t, count=1)
    t = BOM_DIA_RE.sub('', t, count=1)
    candidatos = list(FECHO_RE.finditer(t))
    if candidatos:
        limite = max(int(0.35 * len(t)), len(t) - 2500)
        finais = [m for m in candidatos if m.start() >= limite]
        if finais:
            corte = min(m.start() for m in finais)
            t = t[:corte]
    return t.strip()

def limpar_texto(t, lower=False, clean_edges=True):
    """Pura: NÃO escreve em variáveis globais."""
    if not isinstance(t, str):
        return ""
    t = corrigir_encoding(t)
    t = unicodedata.normalize("NFC", t)
    if clean_edges:
        t = limpar_saudacao_rodape(t)
    t = mascarar_entidades(t)
    t = re.sub(r"\b(\w+)-\n(\w+)", r"\1\2", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t.lower() if lower else t


## 5 — Dataset e collator

- `TextDataset`: tokeniza com **head+tail truncation** — preserva o início (onde ficam a resposta principal) e o fim (onde costuma estar a conclusão), descartando o meio. Melhor que truncar só o fim.
- `SoftLabelCollator`: como `DataCollatorWithPadding`, mas empilha `labels` sem pad. Aceita targets suaves (vetores de probabilidade), não só classes inteiras.

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, soft_targets, tokenizer, max_length=MAX_LENGTH):
        self.texts       = list(texts)
        self.targets     = np.asarray(soft_targets, dtype=np.float32)
        self.tokenizer   = tokenizer
        self.max_length  = max_length
        self.cls_id      = tokenizer.cls_token_id
        self.sep_id      = tokenizer.sep_token_id

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self._encode_head_tail(self.texts[idx])
        enc["labels"] = torch.tensor(self.targets[idx], dtype=torch.float32)
        return enc

    def _encode_head_tail(self, text):
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]
        budget = self.max_length - 2
        if len(ids) > budget:
            half = budget // 2
            ids = ids[:half] + ids[-half:]
        input_ids = [self.cls_id] + list(ids) + [self.sep_id]
        return {
            "input_ids":      input_ids,
            "attention_mask": [1] * len(input_ids),
        }


class SoftLabelCollator:
    """Como DataCollatorWithPadding, mas empilha labels sem tentar padá-los."""
    def __init__(self, tokenizer):
        self.pad_collator = DataCollatorWithPadding(
            tokenizer, padding=True, return_tensors="pt"
        )
    def __call__(self, features):
        labels = [f.pop("labels") for f in features]
        batch  = self.pad_collator(features)
        batch["labels"] = torch.stack(labels, dim=0)   # (B, 3)
        return batch

## 6 — Loss e otimizador

- `soft_ce_loss`: cross-entropy para **targets suaves**. Suporta `label_smoothing` (embaralha os alvos para reduzir overconfidence) e `weight` por classe (lida com desbalanceamento).
- `build_param_groups`: cria três grupos de parâmetros no AdamW com **learning rates distintos**:
  - LoRA (`lora_A`, `lora_B`): LR baixo (3e-5), pois parte de pesos pré-treinados.
  - Classificador (inicializado do zero): LR alto (3e-4), precisa aprender rápido.
  - Outros (fallback): LR médio.

In [ ]:
def soft_ce_loss(logits, soft_targets, weight=None, label_smoothing=0.0):
    """Cross-entropy para targets suaves (probabilidades)."""
    if label_smoothing > 0:
        C = soft_targets.size(-1)
        soft_targets = soft_targets * (1.0 - label_smoothing) + label_smoothing / C
    log_probs = F.log_softmax(logits, dim=-1)
    per_class = -(soft_targets * log_probs)
    if weight is not None:
        per_class = per_class * weight.view(1, -1)
    return per_class.sum(dim=-1).mean()


def build_param_groups(model, lr_lora, lr_classifier, lr_other, weight_decay=0.01):
    """Separa LoRA / classificador / resto para AdamW."""
    lora_p, clf_p, other_p = [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "lora_" in name:
            lora_p.append(p)
        elif "classifier" in name or "modules_to_save" in name:
            clf_p.append(p)
        else:
            other_p.append(p)

    groups = [{"params": lora_p, "lr": lr_lora, "weight_decay": weight_decay}]
    if clf_p:
        groups.append({"params": clf_p, "lr": lr_classifier, "weight_decay": weight_decay})
    if other_p:
        groups.append({"params": other_p, "lr": lr_other, "weight_decay": weight_decay})
    return groups

## 7 — Dados e variantes de pré-processamento

Carrega `train.xlsx`, aplica limpeza e constrói 4 variantes experimentais:

| Variante | Agrupamento | Target |
|---|---|---|
| **A** | prefix=200 (near-dup) | soft labels |
| **B** | hash (exato) | soft labels |
| **C** | prefix=200 | hard majority |
| **D** | prefix=200, só grupos sem conflito | hard majority |

O split é feito com `StratifiedGroupKFold`:
- A/B/C compartilham o split de A (mesmos grupos de treino/val).
- D usa split próprio (subset filtrado).

**Por quê**: comparar se soft labels vs hard labels, e se agrupamento estrito vs frouxo, impactam a acurácia.

In [ ]:
# ============ CARREGAR DADOS ============
dados_base = pd.read_excel(TRAIN_PATH, converters={"resp_text": str, "clarity": str})
dados_base.columns = ["resp_text", "clarity"]
dados_base = dados_base.dropna(subset=["resp_text", "clarity"]).copy()
dados_base["resp_text"] = dados_base["resp_text"].astype(str)
dados_base["clarity"]   = dados_base["clarity"].astype(str).str.strip().str.lower()
dados_base["texto_limpo"] = dados_base["resp_text"].map(limpar_texto)
mask = dados_base["texto_limpo"].str.len() >= MIN_CHARS
dados_base = dados_base[mask].reset_index(drop=True)
print(f"Total: {len(dados_base)}")
print(dados_base["clarity"].value_counts().to_dict())

def hard_from_dist(d):
    max_p = max(d.values())
    cand = [k for k, p in d.items() if p == max_p]
    for c in CLASSES:
        if c in cand: return c
    return cand[0]

def montar_variante(dados_df, grupo_col, usar_soft, nome):
    cont = defaultdict(Counter)
    for g, c in zip(dados_df[grupo_col].tolist(), dados_df["clarity"].tolist()):
        cont[g][c] += 1
    mapa_dist = {g: {k: v/sum(c.values()) for k, v in c.items()}
                 for g, c in cont.items()}
    mapa_hard = {g: hard_from_dist(d) for g, d in mapa_dist.items()}

    y_soft, y_hard, y_true, grupos_out = [], [], [], []
    for _, row in dados_df.iterrows():
        g = row[grupo_col]
        d = mapa_dist[g]
        v = np.array([d.get(c, 0.0) for c in CLASSES], dtype=np.float32)
        if not usar_soft:
            vh = np.zeros(len(CLASSES), dtype=np.float32)
            vh[class_to_idx[mapa_hard[g]]] = 1.0
            v = vh
        y_soft.append(v)
        y_hard.append(mapa_hard[g])
        y_true.append(class_to_idx[row["clarity"]])   # <-- rótulo ORIGINAL da instância
        grupos_out.append(g)

    n_conf = sum(1 for d in mapa_dist.values() if len(d) > 1)
    print(f"\n=== {nome} ===")
    print(f"  N: {len(dados_df)} | Grupos: {len(mapa_dist)} | "
          f"Com conflito: {n_conf}")
    return (dados_df["texto_limpo"].tolist(),
            np.stack(y_soft), y_hard, np.array(y_true), grupos_out)

grupo_A = dados_base["texto_limpo"].map(normalizar_grupo)
grupo_B = dados_base["texto_limpo"].map(hash_grupo)

dados_A = dados_base.copy(); dados_A["grupo"] = grupo_A
dados_B = dados_base.copy(); dados_B["grupo"] = grupo_B

nunique_por_grupo = dados_A.groupby("grupo")["clarity"].nunique()
grupos_limpos = set(nunique_por_grupo[nunique_por_grupo == 1].index)
mask_limpo = dados_A["grupo"].isin(grupos_limpos)
dados_D = dados_A[mask_limpo].reset_index(drop=True)

X_A, Ys_A, Yh_A, Yt_A, G_A = montar_variante(dados_A, "grupo", True,  "A: prefix200+soft")
X_B, Ys_B, Yh_B, Yt_B, G_B = montar_variante(dados_B, "grupo", True,  "B: hash+soft")
X_C, Ys_C, Yh_C, Yt_C, G_C = montar_variante(dados_A, "grupo", False, "C: prefix200+hard")
X_D, Ys_D, Yh_D, Yt_D, G_D = montar_variante(dados_D, "grupo", False, "D: limpo+hard")

# ============ SPLIT COMUM (fold 0) ============
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(sgkf.split(dados_A, Yh_A, groups=grupo_A))
tr_idx_A, va_idx_A = splits[FOLD]

sgkf_D = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits_D = list(sgkf_D.split(dados_D, dados_D["clarity"], groups=dados_D["grupo"]))
tr_idx_D, va_idx_D = splits_D[FOLD]

def subset(X_all, Ys_all, Yh_all, Yt_all, tr, va):
    return ([X_all[i] for i in tr], Ys_all[tr],
            [X_all[i] for i in va], Ys_all[va],
            [Yh_all[i] for i in tr], [Yh_all[i] for i in va],
            Yt_all[tr], Yt_all[va])

VAR = {}
VAR["A"] = subset(X_A, Ys_A, Yh_A, Yt_A, tr_idx_A, va_idx_A)
VAR["B"] = subset(X_B, Ys_B, Yh_B, Yt_B, tr_idx_A, va_idx_A)
VAR["C"] = subset(X_C, Ys_C, Yh_C, Yt_C, tr_idx_A, va_idx_A)
VAR["D"] = subset(X_D, Ys_D, Yh_D, Yt_D, tr_idx_D, va_idx_D)

Total: 19790
{'c5': 6778, 'c234': 6757, 'c1': 6255}

=== A: prefix200+soft ===
  N: 19790 | Grupos: 15823 | Com conflito: 600

=== B: hash+soft ===
  N: 19790 | Grupos: 17118 | Com conflito: 430

=== C: prefix200+hard ===
  N: 19790 | Grupos: 15823 | Com conflito: 600

=== D: limpo+hard ===
  N: 15702 | Grupos: 15223 | Com conflito: 0


## 8 — Baseline oficial reproduzido

Reproduz o `baseline.ipynb` do enunciado: TF-IDF + LogReg com **StratifiedKFold 10 folds** sobre o texto cru.

Serve como referência absoluta (o enunciado exige superar essa acurácia). O valor esperado é ~46,36%.

In [ ]:
X_oficial = dados_base["resp_text"].tolist()
Y_oficial = dados_base["clarity"].tolist()

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
folds_oficial = list(skf.split(X_oficial, Y_oficial))

# Baseline oficial CV (deve dar ~46.36%)
from sklearn.metrics import accuracy_score
accs = []
for tr, va in folds_oficial:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=5,
                                  max_df=0.95, max_features=20000,
                                  sublinear_tf=True, strip_accents="unicode")),
        ("clf", LogisticRegression(C=0.5, class_weight="balanced", max_iter=150,
                                   solver="saga", tol=1e-3, random_state=42)),
    ])
    pipe.fit([X_oficial[i] for i in tr], [Y_oficial[i] for i in tr])
    accs.append(pipe.score([X_oficial[i] for i in va], [Y_oficial[i] for i in va]))
print(f"Baseline oficial CV: {sum(accs)/len(accs):.4f}")  # deve bater ~46.36%

Baseline oficial CV: 0.4617


## 8.1 — Comparação justa com o baseline oficial

O baseline oficial do enunciado é **TF-IDF + Regressão Logística** treinado sobre o **texto cru** com **rótulos originais** em validação cruzada de 10 folds. Esse é o número (~46,36%) que precisa ser superado.

Porém, o baseline calculado na seção anterior é treinado e avaliado **no mesmo split do BERT+LoRA** (fold 0), o que garante um delta honesto, mas não é o número do enunciado. Para não misturar as duas coisas, esta seção calcula **três baselines**:

| Baseline | Texto | Rótulo | Split | Para que serve |
|---|---|---|---|---|
| **`baseline_oficial`** | cru | original | 10 folds próprios (`StratifiedKFold`) | Reproduzir o número do enunciado (~46,36%) como referência absoluta. |
| **`baseline_ABC_split`** | cru | original | mesmo fold 0 do BERT (A/B/C) | Delta justo para A/B/C: mesma partição de treino/validação, mesmo conjunto de instâncias. |
| **`baseline_D_split`** | cru | original | mesmo fold 0 do BERT (D) | Delta justo para D (subset filtrado, split próprio). |

**Por quê os três**: o `delta_oficial` responde à exigência do enunciado ("resultado inferior ao baseline oficial"), mas compara splits diferentes — útil como referência, não como comparação limpa. O `delta_split` é a comparação correta: mesmo conjunto de validação, mesma instância, mesmo rótulo original. É esse número que deve ser destacado no relatório e no slide.

**Ponto importante**: o TF-IDF do baseline sempre usa **texto cru** e **rótulo original**, mesmo quando o BERT+LoRA usa **texto pré-processado** e **alvos suavizados** (soft labels de grupo). Essa assimetria é intencional — o baseline do enunciado é definido assim, e o denoising (soft labels no treino) é a contribuição do nosso modelo, não do baseline.

In [ ]:
def make_tfidf_logreg():
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=5,
                                  max_df=0.95, max_features=20000,
                                  sublinear_tf=True, strip_accents="unicode")),
        ("clf", LogisticRegression(C=0.5, class_weight="balanced", max_iter=150,
                                   solver="saga", tol=1e-3, random_state=42)),
    ])

# --- 1) Baseline OFICIAL: 10-fold CV sobre texto cru + rótulo original ---
X_cru   = dados_base["resp_text"].tolist()
Y_orig  = dados_base["clarity"].tolist()

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
accs_oficial = []
for tr, va in skf.split(X_cru, Y_orig):
    pipe = make_tfidf_logreg()
    pipe.fit([X_cru[i] for i in tr], [Y_orig[i] for i in tr])
    accs_oficial.append(pipe.score([X_cru[i] for i in va], [Y_orig[i] for i in va]))
baseline_oficial = float(np.mean(accs_oficial))
print(f"[Oficial 10-fold CV] acc = {baseline_oficial:.4f}  (esperado ~0.4636)")

# --- 2) Baseline no MESMO split do BERT (fold 0 de A/B/C) ---
# tr_idx_A / va_idx_A são posições em dados_A, que preserva a ordem de dados_base.
X_tr_raw_ABC = [dados_base["resp_text"].iloc[i] for i in tr_idx_A]
X_va_raw_ABC = [dados_base["resp_text"].iloc[i] for i in va_idx_A]
Y_tr_raw_ABC = [dados_base["clarity"].iloc[i] for i in tr_idx_A]   # rótulo ORIGINAL
Y_va_raw_ABC = [dados_base["clarity"].iloc[i] for i in va_idx_A]   # rótulo ORIGINAL

pipe_ABC = make_tfidf_logreg()
pipe_ABC.fit(X_tr_raw_ABC, Y_tr_raw_ABC)
baseline_ABC_split = float(pipe_ABC.score(X_va_raw_ABC, Y_va_raw_ABC))
print(f"[Split A/B/C, texto cru + rótulo original] acc = {baseline_ABC_split:.4f}")

# --- 3) Baseline no MESMO split do BERT (fold 0 de D) ---
# tr_idx_D / va_idx_D são posições em dados_D, que é um subset de dados_A.
# Precisamos mapear de volta para dados_base pelo índice original.
idx_original_D = np.where(mask_limpo.to_numpy())[0]
tr_orig_D = idx_original_D[tr_idx_D]
va_orig_D = idx_original_D[va_idx_D]

X_tr_raw_D = [dados_base["resp_text"].iloc[i] for i in tr_orig_D]
X_va_raw_D = [dados_base["resp_text"].iloc[i] for i in va_orig_D]
Y_tr_raw_D = [dados_base["clarity"].iloc[i] for i in tr_orig_D]
Y_va_raw_D = [dados_base["clarity"].iloc[i] for i in va_orig_D]

pipe_D_raw = make_tfidf_logreg()
pipe_D_raw.fit(X_tr_raw_D, Y_tr_raw_D)
baseline_D_split = float(pipe_D_raw.score(X_va_raw_D, Y_va_raw_D))
print(f"[Split D, texto cru + rótulo original]   acc = {baseline_D_split:.4f}")

print("\nResumo:")
print(f"  baseline_oficial     = {baseline_oficial:.4f}")
print(f"  baseline_ABC_split   = {baseline_ABC_split:.4f}")
print(f"  baseline_D_split     = {baseline_D_split:.4f}")

[Oficial 10-fold CV] acc = 0.4617  (esperado ~0.4636)
[Split A/B/C, texto cru + rótulo original] acc = 0.4685
[Split D, texto cru + rótulo original]   acc = 0.4468

Resumo:
  baseline_oficial     = 0.4617
  baseline_ABC_split   = 0.4685
  baseline_D_split     = 0.4468


## 9 — Função de treino (1 fold)

Treina BERT+LoRA por `epochs` e avalia a cada época.

Detalhes de reprodutibilidade:
- `torch.use_deterministic_algorithms(True, warn_only=True)` + `CUBLAS_WORKSPACE_CONFIG`.
- `num_workers=0` no DataLoader para evitar race condition.
- Validação **sem autocast** (só o treino usa bf16).

Seleção do melhor modelo: **F1 macro** (com tie-break por acurácia). F1 macro é mais informativo que acurácia quando classes podem estar desbalanceadas.

In [ ]:
def treinar_fold(cfg, train_texts, train_soft, val_texts, val_soft, val_true, log):
    set_seed(SEED)
    torch.use_deterministic_algorithms(True, warn_only=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    le = LabelEncoder(); le.fit(CLASSES)
    num_labels = len(le.classes_)

    tokenizer = AutoTokenizer.from_pretrained(cfg["model"])
    base = AutoModelForSequenceClassification.from_pretrained(
        cfg["model"], num_labels=num_labels)
    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"],
        lora_dropout=cfg["lora_dropout"],
        target_modules=cfg["target_modules"],
        modules_to_save=["classifier"],          # <-- também corrige o classificador congelado
    )
    model = get_peft_model(base, lora_cfg).to(device)

    train_ds = TextDataset(train_texts, train_soft, tokenizer, cfg["max_length"])
    val_ds   = TextDataset(val_texts,   val_soft,   tokenizer, cfg["max_length"])
    collator = SoftLabelCollator(tokenizer)

    train_loader = DataLoader(train_ds, batch_size=cfg["batch"], shuffle=True,
                              collate_fn=collator, num_workers=0)
    val_loader   = DataLoader(val_ds, batch_size=cfg["batch"] * 2,
                              collate_fn=collator, num_workers=0)

    y_hard_train = np.argmax(np.asarray(train_soft), axis=1)
    pesos = compute_class_weight("balanced", classes=np.arange(num_labels), y=y_hard_train)
    pesos_t = torch.tensor(pesos, dtype=torch.float, device=device)

    def loss_fn(logits, soft):
        return soft_ce_loss(logits, soft, weight=pesos_t,
                            label_smoothing=cfg["label_smoothing"])

    optimizer = AdamW(build_param_groups(model, cfg["lr_lora"], cfg["lr_clf"], cfg["lr_lora"]),
                      weight_decay=0.01)
    grad_accum = cfg["grad_accum"]
    steps_per_epoch = (len(train_loader) + grad_accum - 1) // grad_accum
    total_steps = steps_per_epoch * cfg["epochs"]
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(WARMUP_PCT * total_steps),
        num_training_steps=total_steps)

    best_f1, best_acc, best_epoch = -1.0, -1.0, -1
    t0 = time.time()

    for epoch in range(cfg["epochs"]):
        model.train(); optimizer.zero_grad()
        epoch_losses = []
        for step, batch in enumerate(train_loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            soft = batch.pop("labels")
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = model(**batch).logits
                loss = loss_fn(logits.float(), soft) / grad_accum
            loss.backward()
            epoch_losses.append(loss.item() * grad_accum)
            if (step + 1) % grad_accum == 0 or (step + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

        model.eval()
        preds = []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                batch.pop("labels")
                logits = model(**batch).logits.float()
                preds.extend(logits.argmax(dim=1).cpu().numpy())

        reais = list(val_true)
        f1 = f1_score(reais, preds, average="macro")
        acc = accuracy_score(reais, preds)
        mcc = matthews_corrcoef(reais, preds)
        lm = float(np.mean(epoch_losses))
        log(f"  Epoch {epoch+1}/{cfg['epochs']} | loss={lm:.4f} | F1={f1:.4f} | Acc={acc:.4f} | MCC={mcc:.4f}")

        if (f1 > best_f1) or (f1 == best_f1 and acc > best_acc):
            best_f1, best_acc, best_epoch = f1, acc, epoch + 1

    elapsed = time.time() - t0
    del model, base, optimizer, scheduler, train_loader, val_loader
    del train_ds, val_ds, collator
    gc.collect(); torch.cuda.empty_cache()

    return {"best_f1": best_f1, "best_acc": best_acc,
            "best_epoch": best_epoch, "elapsed_min": elapsed / 60}

## 11 — Executa as 4 variantes no fold 0

Para cada variante:
1. Treina BERT+LoRA com `CFG_TREINO` usando o alvo suavizado (soft) do grupo.
2. Avalia no mesmo split, cobrando o **rótulo original** de cada instância.
3. Reporta acurácia, F1 macro e dois deltas: `delta_split` (mesmo split do BERT,
   texto cru + rótulo original — comparação limpa) e `delta_oficial` (contra o
   baseline oficial de 10-fold CV).

Os baselines são calculados na **Seção 8.1** e reaproveitados aqui.

In [ ]:
print(f"Baseline oficial (10-fold CV): {baseline_oficial:.4f}")
print(f"Baseline no split A/B/C:       {baseline_ABC_split:.4f}")
print(f"Baseline no split D:           {baseline_D_split:.4f}")

resultados = {}
for nome in ["A", "B", "C", "D"]:
    X_tr, Y_tr_s, X_va, Y_va_s, _, _, _, Yt_va = VAR[nome]
    log = Logger(OUT_DIR / f"variante_{nome}.txt")
    log("=" * 72)
    log(f"VARIANTE {nome} | Treino: {len(X_tr)} | Val: {len(X_va)}")
    log("=" * 72)
    res = treinar_fold(CFG_TREINO, X_tr, Y_tr_s, X_va, Y_va_s, Yt_va, log)

    base = baseline_D_split if nome == "D" else baseline_ABC_split
    delta_split   = res["best_acc"] - base
    delta_oficial = res["best_acc"] - baseline_oficial

    log(f">>> {nome}: acc={res['best_acc']:.4f} | f1={res['best_f1']:.4f} | "
        f"delta_split={delta_split:+.4f} | delta_oficial={delta_oficial:+.4f} | "
        f"ep={res['best_epoch']} | t={res['elapsed_min']:.1f}min")
    log.close()

    resultados[nome] = {
        "acc": res["best_acc"],
        "f1": res["best_f1"],
        "epoch": res["best_epoch"],
        "delta_split": delta_split,
        "delta_oficial": delta_oficial,
        "t_min": res["elapsed_min"],
    }
    gc.collect(); torch.cuda.empty_cache()

df_res = pd.DataFrame(resultados).T.sort_values("acc", ascending=False)
print("\n" + "=" * 80)
print(f"RESULTADO DAS VARIANTES")
print(f"  baseline_oficial   = {baseline_oficial:.4f}")
print(f"  baseline_ABC_split = {baseline_ABC_split:.4f}")
print(f"  baseline_D_split   = {baseline_D_split:.4f}")
print("=" * 80)
print(df_res.to_string())
df_res.to_csv(OUT_DIR / "variantes_resultados.csv")

Baseline oficial (10-fold CV): 0.4617
Baseline no split A/B/C:       0.4685
Baseline no split D:           0.4468
VARIANTE A | Treino: 17553 | Val: 2237


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

  Epoch 1/5 | loss=1.0832 | F1=0.4366 | Acc=0.4479 | MCC=0.1723
  Epoch 2/5 | loss=1.0553 | F1=0.4135 | Acc=0.4399 | MCC=0.1647
  Epoch 3/5 | loss=1.0374 | F1=0.4457 | Acc=0.4461 | MCC=0.1731
  Epoch 4/5 | loss=1.0206 | F1=0.4380 | Acc=0.4466 | MCC=0.1694
  Epoch 5/5 | loss=1.0118 | F1=0.4538 | Acc=0.4560 | MCC=0.1834
>>> A: acc=0.4560 | f1=0.4538 | delta_split=-0.0125 | delta_oficial=-0.0057 | ep=5 | t=5.1min
VARIANTE B | Treino: 17553 | Val: 2237


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

  Epoch 1/5 | loss=1.0578 | F1=0.4247 | Acc=0.4341 | MCC=0.1513
  Epoch 2/5 | loss=1.0084 | F1=0.4412 | Acc=0.4497 | MCC=0.1764
  Epoch 3/5 | loss=0.9797 | F1=0.4447 | Acc=0.4448 | MCC=0.1708
  Epoch 4/5 | loss=0.9534 | F1=0.4475 | Acc=0.4497 | MCC=0.1742
  Epoch 5/5 | loss=0.9386 | F1=0.4485 | Acc=0.4511 | MCC=0.1765
>>> C: acc=0.4511 | f1=0.4485 | delta_split=-0.0174 | delta_oficial=-0.0106 | ep=5 | t=5.0min
VARIANTE D | Treino: 14122 | Val: 1580


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

  Epoch 1/5 | loss=1.0876 | F1=0.4159 | Acc=0.4253 | MCC=0.1292
  Epoch 2/5 | loss=1.0486 | F1=0.4461 | Acc=0.4576 | MCC=0.1880
  Epoch 3/5 | loss=1.0263 | F1=0.4356 | Acc=0.4418 | MCC=0.1632
  Epoch 4/5 | loss=1.0080 | F1=0.4410 | Acc=0.4475 | MCC=0.1652
  Epoch 5/5 | loss=0.9938 | F1=0.4420 | Acc=0.4525 | MCC=0.1786
>>> D: acc=0.4576 | f1=0.4461 | delta_split=+0.0108 | delta_oficial=-0.0041 | ep=2 | t=4.0min

RESULTADO DAS VARIANTES
  baseline_oficial   = 0.4617
  baseline_ABC_split = 0.4685
  baseline_D_split   = 0.4468
        acc        f1  epoch  delta_split  delta_oficial     t_min
B  0.463120  0.456096    4.0    -0.005364       0.001422  5.034367
D  0.457595  0.446055    2.0     0.010759      -0.004103  4.015816
A  0.455968  0.453835    5.0    -0.012517      -0.005730  5.053819
C  0.451051  0.448488    5.0    -0.017434      -0.010647  5.034439
